In [1]:
from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessModels.DeclareModel import DeclareModelTemplate
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser


import pickle, os
import pandas as pd
import numpy as np


In [2]:
event_log_name = "huge"
log_path = f"./.out/eventlogs/{event_log_name}-0.3-1.xes"

event_log = D4PyEventLog(case_name="case:concept:name")
event_log.parse_xes_log(log_path)

/home/devashish/anaconda3/envs/ltn/lib/python3.9/site-packages/pm4py/util/dt_parsing/parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/5000 [00:00<?, ?it/s]

In [3]:
# Save the conformance checking results to disk to avoid recalculating
def save_conformance_results(conf_check_res, filename=f'{event_log_name}_5000_conformance_results.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(conf_check_res, f)
    print(f"Conformance checking results saved to {filename}")

# Load the conformance checking results from disk
def load_conformance_results(filename=f'{event_log_name}_5000_conformance_results.pkl'):
    try:
        with open(filename, 'rb') as f:
            conf_check_res = pickle.load(f)
        print(f"Conformance checking results loaded from {filename}")
        return conf_check_res
    except FileNotFoundError:
        print(f"File {filename} not found. Run conformance checking first.")
        return None

In [4]:
if not os.path.exists(f"{event_log_name}-0.3-1.decl"):
    print(f"File {event_log_name}-0.3-1.decl does not exist, running discovery...")
    discovery = DeclareMiner(log=event_log, consider_vacuity=False, min_support=0.05, itemsets_support=0.05, max_declare_cardinality=1)
    declare_model: DeclareModel = discovery.run()
    print(f"Total constraints discovered: {len(declare_model.serialized_constraints)}")
    model_constraints = declare_model.get_decl_model_constraints()
    declare_model.to_file(f"{event_log_name}-0.3-1.decl")

In [ ]:
if not os.path.exists(f'{event_log_name}_5000_conformance_results.pkl') and event_log is not None and declare_model is not None:
    print(f"File {event_log_name}_5000_conformance_results.pkl does not exist, running conformance checking...")
    basic_checker = MPDeclareAnalyzer(log=event_log, declare_model=declare_model, consider_vacuity=False)
    conf_check_res: MPDeclareResultsBrowser = basic_checker.run()
    save_conformance_results(conf_check_res)
else:
    print(f"Loading conformance checking results from {event_log_name}_5000_conformance_results.pkl")
    conf_check_res = load_conformance_results()
conf_check_df =  conf_check_res.get_metric(metric="state")


Loading conformance checking results from huge_5000_conformance_results.pkl


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x731b9da79b20>>
Traceback (most recent call last):
  File "/home/devashish/anaconda3/envs/ltn/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [ ]:
activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)
satisfied_df = conf_check_res.get_metric(metric="state").fillna(0)

# Support = how often constraint is satisfied
support = satisfied_df.sum(axis=0) / len(satisfied_df)

# Activation rate = how often constraint is activated
activated_counts = activated_df.sum(axis=0)
activation_rate = activated_counts / len(activated_df)

# Satisfied counts
satisfied_counts = satisfied_df.sum(axis=0)

# Confidence = P(satisfied | activated), safe division
confidence = np.where(
    activated_counts != 0,
    satisfied_counts / activated_counts,
    0
)
confidence = pd.Series(confidence, index=activated_counts.index)

# Combine into a DataFrame
metrics_df = pd.DataFrame({
    'support': support,
    'confidence': confidence,
    # 'activation_rate': activation_rate
})
# metrics_df


/tmp/ipykernel_1980216/1810640346.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)


In [ ]:
# filtered_metrics_df is metrics_df but with rows with support less than 0.2 and in descending order of confidence and no "not" in the constraint name
filtered_metrics_df = metrics_df[~metrics_df.index.str.contains("Not") & (metrics_df['support'] <= 0.2) & (metrics_df['confidence'] >= 0.7)].sort_values(by='confidence', ascending=False)
# filtered_metrics_df = metrics_df[metrics_df['support'] <= 0.2].sort_values(by='confidence', ascending=False)
print("Filtered Metrics DataFrame:")
display(filtered_metrics_df)

Filtered Metrics DataFrame:


,support,confidence
"Response[Activity AD, Activity B] | |",0.0652,0.996942
"Responded Existence[Activity AD, Activity B] | |",0.0652,0.996942
"Alternate Response[Activity AD, Activity B] | |",0.0650,0.993884
"Responded Existence[Activity AD, Activity S] | |",0.0650,0.993884
"Responded Existence[Activity AM, Activity B] | |",0.1284,0.993808
...,...,...
"Alternate Precedence[Activity M, Activity AC] | |",0.0598,0.911585
"Alternate Precedence[Activity AM, Activity AO] | |",0.1194,0.911450
"Alternate Precedence[Activity AI, Activity AO] | |",0.1194,0.911450
"Alternate Precedence[Activity AL, Activity AO] | |",0.1190,0.908397


In [ ]:
raise KeyboardInterrupt("\nStopping execution after displaying filtered metrics DataFrame.\nChoose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.\nThen run the cells below again to see the results of the selected constraints.")

KeyboardInterrupt: 
Stopping execution after displaying filtered metrics DataFrame.
Choose your favourite constraints from the filtered_metrics_df DataFrame and add them to the interesting_constraints list in the ipynb.
Then run the cells below again to see the results of the selected constraints.

# Constraints with low support and high confidence
1. Responded Existence[Activity AM, Activity B] | |	0.1284	0.9938080495356038

In [ ]:
interesting_constraints = [
    # "Response[Approve PO 2, Release PO] | |",
    "Responded Existence[Activity AM, Activity B] | |"
    ]

In [ ]:
state_df = conf_check_res.get_metric("state")[interesting_constraints]
non_zero_counts = state_df.ne(0).sum(axis=0)
print(non_zero_counts)
non_zero_rows = state_df.index[state_df[interesting_constraints[0]] != 0].tolist()
print(non_zero_rows)
print(len(non_zero_rows))
with open(f'{event_log_name}_ltn_rows.pkl', 'wb') as f:
    pickle.dump(non_zero_rows, f)

Responded Existence[Activity AM, Activity B] | |    642
dtype: int64
[0, 3, 13, 15, 25, 27, 28, 33, 42, 49, 51, 65, 81, 88, 92, 94, 100, 112, 115, 116, 119, 137, 157, 159, 160, 164, 189, 191, 196, 208, 212, 215, 220, 221, 226, 230, 260, 262, 294, 296, 307, 315, 326, 333, 343, 345, 353, 358, 368, 386, 402, 419, 421, 437, 439, 440, 447, 456, 477, 478, 486, 487, 495, 501, 510, 511, 516, 524, 534, 536, 545, 550, 564, 565, 573, 580, 588, 590, 594, 601, 603, 604, 621, 630, 632, 643, 644, 648, 649, 660, 669, 674, 684, 687, 689, 690, 695, 702, 707, 709, 716, 724, 732, 748, 754, 756, 767, 770, 791, 809, 812, 832, 852, 854, 857, 871, 875, 889, 913, 920, 921, 960, 971, 979, 985, 989, 994, 1031, 1038, 1039, 1046, 1061, 1068, 1070, 1080, 1081, 1086, 1102, 1112, 1118, 1128, 1131, 1138, 1151, 1158, 1164, 1174, 1178, 1184, 1189, 1192, 1197, 1212, 1219, 1225, 1238, 1251, 1259, 1263, 1277, 1287, 1298, 1303, 1314, 1321, 1324, 1331, 1366, 1370, 1373, 1374, 1381, 1389, 1406, 1412, 1418, 1425, 1426, 1427, 1

In [ ]:
print("END")

END


In [ ]:
# summary_df = conf_check_df.apply(lambda col: col.value_counts()).fillna(0).astype(int)
# summary_df = summary_df.reindex([0, 1])
# summary_df = summary_df / len(conf_check_df)
# summary_df = summary_df.T
# summary_df = summary_df.sort_values(by=1, ascending=False)
# display(summary_df)